In [4]:
import pandas as pd
import nltk 
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy
from scipy.sparse import hstack
import numpy as np
import spacy
import re

In [ ]:
#punkt è modello di tokenizzatore
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet = True)
#POS tokenizer per riconoscere grammaticalmente le parti dl discorso
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

True

In [ ]:
#Load data
df = pd.read_csv('../Dati/Raw/articles_clean.csv')

#Funzione per selezionare il primo quartile di un articolo (v. paper reference)
def truncate_to_25pct(text):
    if not isinstance(text, str): return ""
    sentences = nltk.sent_tokenize(text) #Tokenizzazione per frasi (sentences)
    limit = max(1, len(sentences) // 4)
    return " ".join(sentences[:limit])

#df.head()


,article_id,topic_id,date,original_orientation,binary_label,full_text
0,61d9f869-d054-4938-9c9c-6351777db2ad,"Powerful Earthquakes Hit Venezuela, USGS Estim...",NaN,left,1,"More than 900 people have been injured, but th..."
1,f1bb9659-50f4-4521-930e-c2158a2281af,"Powerful Earthquakes Hit Venezuela, USGS Estim...",NaN,center,0,Tremors Kill At Least 164 People in Venezuela ...
2,d230406b-a562-4d90-b5d4-5f3ed51fd256,"Powerful Earthquakes Hit Venezuela, USGS Estim...",NaN,right,1,Venezuela was rocked Wednesday afternoon by it...
3,e1787ee3-88e4-4156-adb1-32a097da0a16,Supreme Court Rules Prison Guards Cannot Be Su...,NaN,left,1,Exterior view of the U.S. Supreme Court Buildi...
4,6e6aa0bb-f209-42dc-8918-4c1a056d9f8b,Supreme Court Rules Prison Guards Cannot Be Su...,NaN,center,0,The US Supreme Court has ruled that a former L...


In [ ]:
#Trovo articoli troppo corti ed elimino l'intera tripletta di cui fa parte
df['word_count'] = df['full_text'].str.split().str.len()
invalid_topics = df[df['word_count'] < 50]['topic_id'].unique()  #per individuare la tripletta
df_clean = df[~df['topic_id'].isin(invalid_topics)]

#Sanity check
print(f"Topics removed: {len(invalid_topics)}")
print(f"Residual topics: {df_clean['topic_id'].nunique()}")
df['full_text'] = df['full_text'].apply(truncate_to_25pct)

#Save
df.to_csv('articles_25%_sentences.csv', index=False)

#Load truncated data
df = pd.read_csv('articles_25%_sentences.csv')

df['full_text'] = df['full_text'].replace(r'\s+', ' ', regex=True).str.strip()

Topics removed: 5
Residual topics: 64


In [ ]:
#spaCy per fare riconoscere entità, organizzazioni (entities), tutti termini difficilmente presenti in un dizionaio standard
nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"]) #Assumiamo sia già installato su si usa environment

def mask_named_entities(text):
    doc = nlp(text)
    #Replace every named entity with label (e.g., PERSON, ORG,...)
    for ent in reversed(doc.ents):
        text = text[:ent.start_char] + ent.label_ + text[ent.end_char:]
    return text

In [ ]:
#Applica la mask appena costruita per le 'entities'
df['full_text'] = df['full_text'].apply(mask_named_entities)

In [13]:
def rigorous_clean(text):
    text = str(text)
    #Remove everything except alphanumeric, spaces, and basic punctuation
    text = re.sub(r'[^\w\s.,!?]', '', text)
    #Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [14]:
df['full_text'] = df['full_text'].apply(rigorous_clean)

In [15]:
# POS Tagging extraction function --> cioè riconosce verbo, soggetto, ecc...
def get_pos_tags(text):
    tokens = nltk.word_tokenize(text)
    tags = nltk.pos_tag(tokens)
    return " ".join([tag for word, tag in tags])

In [16]:
#Extract POS tags
df['pos_text'] = df['full_text'].apply(get_pos_tags)

In [18]:
#Usiamo gli n-grammi (2-4) come struttura di codifica
#In particolare usiamo TF-IDF per creare features stilometriche (v. paper)
char_vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), min_df=0.05, strip_accents='unicode')
char_features = char_vectorizer.fit_transform(df['full_text'].astype(str))

#N-grammi (1-2) sulla struttura del discorso (POS)
pos_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=0.05) #minimo di frequenza degli n-grammi
pos_features = pos_vectorizer.fit_transform(df['pos_text'])

X = hstack([char_features, pos_features])

print(f"Total number of stylometric features: {X.shape[1]}")

Total number of stylometric features: 12940


In [22]:
#Obiettivo: vedere, in base a TF-IDF quanto sono prominenti le strutture (suona meglio in inglese ma ok), 
#sia a livello di caratteri (char) che a livello di parti del discorso (pos)

#Uniamo dizionari char e pos
char_names = char_vectorizer.get_feature_names_out()
pos_names = pos_vectorizer.get_feature_names_out()
all_names = np.concatenate([char_names, pos_names])

#TF-IDF Medio across i vari articoli
mean_scores = np.array(X.mean(axis=0)).flatten()

#DataFrame per visualizzare risultati
feature_prominence = pd.DataFrame({
    'feature': all_names,
    'prominence': mean_scores
}).sort_values(by='prominence', ascending=False)

print(feature_prominence.head(20))

      feature  prominence
12644      nn    0.439171
12670     nnp    0.418077
12583      in    0.378207
12557      dt    0.285730
3228       e     0.221964
12697     nns    0.190438
12607      jj    0.181651
863         t    0.181502
17          a    0.150735
12814     vbd    0.140943
10094      s     0.140414
11122      th    0.139832
5749       in    0.138810
12565   dt nn    0.135928
12648   nn in    0.128245
2761       d     0.118947
12723     prp    0.118606
5174       he    0.117269
640         o    0.116513
12586   in dt    0.114845
